<a href="https://colab.research.google.com/github/yeshaa23/Project-A-Kelompok-4-Pertamina-PBAGenap/blob/main/Notebook/2a_Prepocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# MOUNT GOOGLE DRIVE
# ============================================================
from google.colab import drive

# Menghubungkan Google Drive dengan Colab
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# ============================================================
# MEMUAT DATASET DARI GOOGLE DRIVE
# ============================================================
import pandas as pd

# Ganti dengan path file yang sesuai di Google Drive Anda
file_path = '/content/drive/MyDrive/ProjectA-PBA/hasil_scraping_berita.csv'

# Memuat data dari file CSV
df = pd.read_csv(file_path)

# Tampilkan beberapa baris pertama untuk memastikan data dimuat dengan benar
df.head()

,Link,Judul,Isi Berita,Status,Tag,Sentimen,Penerbit
0,https://money.kompas.com/image/2017/10/03/1815...,Foto : CSR Pertamina Lubricants Kini Fokus ke ...,Diskusi mengenai CSR di industri pelumas berta...,success,Ekonomi,Netral,money.kompas.com
1,https://money.kompas.com/read/2016/06/06/14280...,"Gandeng Dua Bank, Pertamina Lubricants Yakin P...","JAKARTA, KOMPAS.com - Anak perusahaan Pertamin...",success,Kecelakaan Kerja,Positif,money.kompas.com
2,https://money.kompas.com/image/2017/08/03/0900...,"Foto : Cegah Kepunahan, Pertamina Lestarikan T...",Pelepasan Indukan Tuntong Laut oleh tim konser...,success,Bisnis,Positif,money.kompas.com
3,https://money.kompas.com/read/2016/04/11/17345...,Pertamina Menaikkan Ongkos Angkut Minyak Penam...,"BOJONEGORO, KOMPAS.com - Pemerintah terus meng...",success,Migas,Positif,money.kompas.com
4,https://money.kompas.com/image/2017/11/27/1656...,Foto : Ini Dua Fokus CSR Pertamina Patra Niaga...,Direktur Operasional PT Pertamina Patra Niaga ...,success,Akademik,Positif,money.kompas.com


In [3]:
# ============================================================
# IMPORT LIBRARY YANG DIBUTUHKAN
# ============================================================
!pip install tqdm
!pip install Sastrawi
import re
import os
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from textblob import TextBlob
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.drawing.image import Image as XLImage
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Mengabaikan peringatan
warnings.filterwarnings("ignore")

In [4]:
# ============================================================
# MEMUAT DATA BERITA PERTAMINA & PREPROCESSING DASAR
# ============================================================
print("=" * 60)
print("  PREPROCESSING BERITA PERTAMINA")
print("=" * 60)

# Memuat dataset hasil scraping
df = pd.read_csv(file_path, encoding="utf-8")  # bisa df_raw juga kalau mau tetap
print(f" Data dimuat: {len(df)} baris, {df.shape[1]} kolom")
print(f"\nKolom yang tersedia: {list(df.columns)}")

# ============================================================
# Mapping Sentimen ke English
# ============================================================
sentiment_map = {'Positif':'Positive', 'Negatif':'Negative', 'Netral':'Neutral'}
df['Sentimen'] = (
    df['Sentimen']
    .astype(str)
    .str.strip()
    .str.capitalize()
    .map(sentiment_map)
    .fillna(df['Sentimen'])
)

# ============================================================
# Normalisasi kolom Penerbit
# ============================================================
# Ubah semua penerbit menjadi Title Case + hapus spasi berlebih
df['Penerbit'] = df['Penerbit'].str.title().str.strip()

# Fungsi normalisasi publisher
def normalize_publisher(name):
    if not isinstance(name, str):
        return name

    name = name.strip().lower()  # lowercase + hapus spasi

    # Mapping domain/keyword
    if 'detik' in name:
        return 'Detik'
    elif 'kompas' in name:
        return 'Kompas'
    elif 'antara' in name:
        return 'Antara News'
    elif 'tempo' in name:
        return 'Tempo'
    elif 'kontan' in name:
        return 'Kontan'
    elif 'bisnis' in name:
        return 'Bisnis'
    else:
        return name.title()  # default: title case nama asli

# Terapkan ke kolom Penerbit
df['Penerbit'] = df['Penerbit'].apply(normalize_publisher)

display(df.head(10))

  PREPROCESSING BERITA PERTAMINA
 Data dimuat: 1557 baris, 7 kolom

Kolom yang tersedia: ['Link', 'Judul', 'Isi Berita', 'Status', 'Tag', 'Sentimen', 'Penerbit']


,Link,Judul,Isi Berita,Status,Tag,Sentimen,Penerbit
0,https://money.kompas.com/image/2017/10/03/1815...,Foto : CSR Pertamina Lubricants Kini Fokus ke ...,Diskusi mengenai CSR di industri pelumas berta...,success,Ekonomi,Neutral,Kompas
1,https://money.kompas.com/read/2016/06/06/14280...,"Gandeng Dua Bank, Pertamina Lubricants Yakin P...","JAKARTA, KOMPAS.com - Anak perusahaan Pertamin...",success,Kecelakaan Kerja,Positive,Kompas
2,https://money.kompas.com/image/2017/08/03/0900...,"Foto : Cegah Kepunahan, Pertamina Lestarikan T...",Pelepasan Indukan Tuntong Laut oleh tim konser...,success,Bisnis,Positive,Kompas
3,https://money.kompas.com/read/2016/04/11/17345...,Pertamina Menaikkan Ongkos Angkut Minyak Penam...,"BOJONEGORO, KOMPAS.com - Pemerintah terus meng...",success,Migas,Positive,Kompas
4,https://money.kompas.com/image/2017/11/27/1656...,Foto : Ini Dua Fokus CSR Pertamina Patra Niaga...,Direktur Operasional PT Pertamina Patra Niaga ...,success,Akademik,Positive,Kompas
5,https://money.kompas.com/image/2016/11/01/1929...,Foto : Pertamina Tegaskan Kelangkaan BBM Hanya...,"Unduh Kompas.com App untuk berita terkini, aku...",success,BBM,Negative,Kompas
6,https://nasional.kompas.com/read/2017/06/09/20...,Mantan Dirut Pertamina Transkontinental Diteta...,"JAKARTA, KOMPAS.com - Jaksa Agung Muda Pidana ...",success,Hukum,Neutral,Kompas
7,https://otomotif.kompas.com/read/2017/09/16/16...,"Setelah Tol, Beli BBM Juga Wajib Pakai Uang El...","Jakarta, KompasOtomotif - Selain PT Jasa Marga...",success,BBM,Negative,Kompas
8,https://otomotif.kompas.com/read/2017/12/13/11...,Pertamax Turbo buat Konsumen Pemilik Supercar,"Nusa Dua, KompasOtomotif - Pertamina kini teng...",success,Kebakaran,Neutral,Kompas
9,https://money.kompas.com/read/2017/02/11/17000...,PHE WMO Pasok Kondensat ke Industri Thinner da...,"GRESIK, KOMPAS - PT Pertamina Hulu Energi West...",success,Migas,Negative,Kompas


In [5]:
# ============================================================
# INISIALISASI NLP: Stopword dan Stemmer
# ============================================================
stemmer_factory = StemmerFactory()
stemmer         = stemmer_factory.create_stemmer()

sw_factory   = StopWordRemoverFactory()
STOPWORDS_ID = set(sw_factory.get_stop_words()) | {
    # Slang & kata umum tidak informatif
    "yg", "nya", "juga", "bisa", "itu", "ini", "ada", "jadi", "dari", "ke", "di", "dan", "atau", "dengan",
    "untuk", "pada", "sudah", "akan", "telah", "saat", "karena", "namun", "tapi", "kalau", "bila", "jika",
    "sehingga", "agar", "kami", "kita", "mereka", "dia", "ia", "anda", "saya", "kamu", "belum", "pun", "pula",
    "lagi", "kini", "hal", "kata", "ujar", "tutur", "tambah", "ungkap", "terang", "sebut", "jelaskan", "menyebut",
    "menurut", "berdasarkan", "yakni", "yaitu", "antara", "lain", "berbagai", "beberapa",
    # Noise scraping
    "http", "https", "www", "com", "id", "co", "go", "baca", "juga", "copyright", "editor", "pewarta",
    "related", "artikel", "berita", "read", "more", "click", "foto", "ilustrasi", "dok", "dokumentasi",
    "caption", "sumber", "gambar",
    # Kata sangat umum di berita
    "tahun", "hari", "bulan", "persen", "rp", "juta", "miliar", "triliun", "pt", "tbk", "persero", "presiden",
    "direktur", "menteri", "kepala",
}

# ============================================================
# DEFINISIKAN FUNGSI PENGOLAHAN TEKS
# ============================================================
def to_uppercase(text: str) -> str:
    return str(text).upper()

def to_lowercase(text: str) -> str:
    return str(text).lower()

def clean_text(text: str) -> str:
    """Membersihkan teks: menghapus URL, HTML, angka, simbol, dan kata-kata pendek."""
    text = re.sub(r"http\S+|www\S+", " ", text)        # Hapus URL
    text = re.sub(r"<[^>]+>", " ", text)               # Hapus tag HTML
    text = re.sub(r"[^\w\s]", " ", text)               # Hapus tanda baca & simbol
    text = re.sub(r"\d+", " ", text)                   # Hapus angka
    text = re.sub(r"\b\w{1,2}\b", " ", text)           # Hapus kata ≤2 karakter
    text = re.sub(r"\s+", " ", text).strip()           # Normalisasi spasi
    return text

def remove_news_boilerplate(text: str) -> str:
    # Hapus kata-kata media
    text = re.sub(r'\b(kompas|cnn|detik|tempo|tribunnews|kontan|bisnis)\b', ' ', text, flags=re.IGNORECASE)
    # Hapus kata umum / ekstensi
    text = re.sub(r'\b(com|co|id|news|finance|money|otomotif)\b', ' ', text, flags=re.IGNORECASE)
    # Hapus beberapa kota / nama negara
    text = re.sub(r'\b(jakarta|indonesia|bojonegoro|surabaya)\b', ' ', text, flags=re.IGNORECASE)
    # Hapus perusahaan / kata lain
    text = re.sub(r'\b(pt|persero|tbk)\b', ' ', text, flags=re.IGNORECASE)
    # Hapus spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def remove_stopwords(text: str) -> str:
    """Menghapus stopwords dari teks."""
    return " ".join(w for w in text.split() if w not in STOPWORDS_ID)

def tokenize(text: str) -> list:
    """Melakukan tokenisasi teks menjadi daftar kata."""
    return [w for w in text.split() if w]

def stem_text(text: str) -> str:
    """Melakukan stemming untuk teks."""
    return " ".join(stemmer.stem(w) for w in text.split())

print("Fungsi preprocessing siap!")

Fungsi preprocessing siap!


In [6]:
# ============================================================
# PIPELINE PREPROCESSING
# ============================================================
from IPython.display import display
from tqdm import tqdm

# 1. Uppercase & Lowercase
print("[1/8] Uppercase & Lowercase...")
df["text_uppercase"] = [to_uppercase(text) for text in tqdm(df["Isi Berita"], desc="Uppercase")]
df["text_lowercase"] = [to_lowercase(text) for text in tqdm(df["Isi Berita"], desc="Lowercase")]
display(df[["Isi Berita", "text_uppercase", "text_lowercase"]].head(5))

# 2. Cleaning teks
print("[2/8] Cleaning teks...")
df["text_clean"] = [clean_text(text) for text in tqdm(df["text_lowercase"], desc="Cleaning")]
display(df[["text_lowercase", "text_clean"]].head(5))

# 3. Menghapus boilerplate
print("[3/8] Menghapus boilerplate berita...")
df["text_no_boilerplate"] = [remove_news_boilerplate(text) for text in tqdm(df["text_clean"], desc="Boilerplate Removal")]
display(df[["text_clean", "text_no_boilerplate"]].head(5))

# 4. Tokenisasi sebelum stopwords
print("[4/8] Tokenisasi sebelum stopwords...")
df["tokens_before_sw"] = [tokenize(text) for text in tqdm(df["text_no_boilerplate"], desc="Tokenize Before SW")]
df["jumlah_token_before_sw"] = df["tokens_before_sw"].apply(len)
display(df[["text_no_boilerplate", "tokens_before_sw", "jumlah_token_before_sw"]].head(5))

# 5. Stopwords removal
print("[5/8] Stopwords removal...")
df["text_no_stopwords"] = [remove_stopwords(text) for text in tqdm(df["text_no_boilerplate"], desc="Stopwords Removal")]
display(df[["text_no_boilerplate", "text_no_stopwords"]].head(5))

# 6. Tokenisasi setelah stopwords
print("[6/8] Tokenisasi setelah stopwords...")
df["tokens_after_sw"] = [tokenize(text) for text in tqdm(df["text_no_stopwords"], desc="Tokenize After SW")]
df["jumlah_token_after_sw"] = df["tokens_after_sw"].apply(len)
display(df[["text_no_stopwords", "tokens_after_sw", "jumlah_token_after_sw"]].head(5))

# 7. Format tokenisasi untuk Excel
print("[7/8] Format tokenisasi...")
df["tokenisasi"] = [" | ".join(tokens) for tokens in tqdm(df["tokens_after_sw"], desc="Format Tokens")]
display(df[["tokens_after_sw", "tokenisasi"]].head(5))

# 8. Stemming
print("[8/8] Stemming...")
df["text_stemmed"] = [stem_text(text) for text in tqdm(df["text_no_stopwords"], desc="Stemming")]
df["tokens_stemmed"] = [tokenize(text) for text in tqdm(df["text_stemmed"], desc="Tokenize Stemmed")]
display(df[["text_no_stopwords", "text_stemmed", "tokens_stemmed"]].head(5))



[1/8] Uppercase & Lowercase...


Lowercase: 100%|██████████| 1557/1557 [00:00<00:00, 55548.73it/s]


,Isi Berita,text_uppercase,text_lowercase
0,Diskusi mengenai CSR di industri pelumas berta...,DISKUSI MENGENAI CSR DI INDUSTRI PELUMAS BERTA...,diskusi mengenai csr di industri pelumas berta...
1,"JAKARTA, KOMPAS.com - Anak perusahaan Pertamin...","JAKARTA, KOMPAS.COM - ANAK PERUSAHAAN PERTAMIN...","jakarta, kompas.com - anak perusahaan pertamin..."
2,Pelepasan Indukan Tuntong Laut oleh tim konser...,PELEPASAN INDUKAN TUNTONG LAUT OLEH TIM KONSER...,pelepasan indukan tuntong laut oleh tim konser...
3,"BOJONEGORO, KOMPAS.com - Pemerintah terus meng...","BOJONEGORO, KOMPAS.COM - PEMERINTAH TERUS MENG...","bojonegoro, kompas.com - pemerintah terus meng..."
4,Direktur Operasional PT Pertamina Patra Niaga ...,DIREKTUR OPERASIONAL PT PERTAMINA PATRA NIAGA ...,direktur operasional pt pertamina patra niaga ...


[2/8] Cleaning teks...


Cleaning: 100%|██████████| 1557/1557 [00:00<00:00, 1822.65it/s]


...

Format Tokens: 100%|██████████| 1557/1557 [00:00<00:00, 34612.96it/s]


,tokens_after_sw,tokenisasi
0,"[diskusi, mengenai, csr, industri, pelumas, be...",diskusi | mengenai | csr | industri | pelumas ...
1,"[anak, perusahaan, pertamina, pertamina, lubri...",anak | perusahaan | pertamina | pertamina | lu...
2,"[pelepasan, indukan, tuntong, laut, tim, konse...",pelepasan | indukan | tuntong | laut | tim | k...
3,"[pemerintah, terus, mengupayakan, penambahan, ...",pemerintah | terus | mengupayakan | penambahan...
4,"[operasional, pertamina, patra, niaga, abdul, ...",operasional | pertamina | patra | niaga | abdu...


[8/8] Stemming...


Tokenize Stemmed: 100%|██████████| 1557/1557 [00:00<00:00, 29858.77it/s]


,text_no_stopwords,text_stemmed,tokens_stemmed
0,diskusi mengenai csr industri pelumas bertajuk...,diskusi kena csr industri lumas tajuk cari for...,"[diskusi, kena, csr, industri, lumas, tajuk, c..."
1,anak perusahaan pertamina pertamina lubricants...,anak usaha pertamina pertamina lubricants gand...,"[anak, usaha, pertamina, pertamina, lubricants..."
2,pelepasan indukan tuntong laut tim konservasi ...,lepas indu tuntong laut tim konservasi alam pe...,"[lepas, indu, tuntong, laut, tim, konservasi, ..."
3,pemerintah terus mengupayakan penambahan produ...,perintah terus upaya tambah produksi minyak ga...,"[perintah, terus, upaya, tambah, produksi, min..."
4,operasional pertamina patra niaga abdul cholid...,operasional pertamina patra niaga abdul cholid...,"[operasional, pertamina, patra, niaga, abdul, ..."


In [13]:
# ============================================================
# PREVIEW HASIL AKHIR
# ============================================================
df["final_text_clean"] = df["text_clean"]            # hasil clean text
df["final_text_stemmed"] = df["text_stemmed"]       # hasil stemmed

# Preview hasil akhir (tabel ringkas)
display(df[["Judul", "final_text_clean", "final_text_stemmed",
            "jumlah_token_before_sw", "jumlah_token_after_sw"]].head(10))

,Judul,final_text_clean,final_text_stemmed,jumlah_token_before_sw,jumlah_token_after_sw
0,Foto : CSR Pertamina Lubricants Kini Fokus ke ...,diskusi mengenai csr industri pelumas bertajuk...,diskusi kena csr industri lumas tajuk cari for...,16,13
1,"Gandeng Dua Bank, Pertamina Lubricants Yakin P...",jakarta kompas com anak perusahaan pertamina y...,anak usaha pertamina pertamina lubricants gand...,130,95
2,"Foto : Cegah Kepunahan, Pertamina Lestarikan T...",pelepasan indukan tuntong laut oleh tim konser...,lepas indu tuntong laut tim konservasi alam pe...,34,31
3,Pertamina Menaikkan Ongkos Angkut Minyak Penam...,bojonegoro kompas com pemerintah terus mengupa...,perintah terus upaya tambah produksi minyak ga...,302,224
4,Foto : Ini Dua Fokus CSR Pertamina Patra Niaga...,direktur operasional pertamina patra niaga abd...,operasional pertamina patra niaga abdul cholid...,38,32
5,Foto : Pertamina Tegaskan Kelangkaan BBM Hanya...,unduh kompas com app untuk berita terkini akur...,unduh app kini akurat percaya arah kamera kode...,17,10
6,Mantan Dirut Pertamina Transkontinental Diteta...,jakarta kompas com jaksa agung muda pidana khu...,jaksa agung muda pidana khusus jaksa agung tet...,145,104
7,"Setelah Tol, Beli BBM Juga Wajib Pakai Uang El...",jakarta kompasotomotif selain jasa marga perse...,kompasotomotif jasa marga pertamina siap laku ...,290,225
8,Pertamax Turbo buat Konsumen Pemilik Supercar,nusa dua kompasotomotif pertamina kini tengah ...,nusa kompasotomotif pertamina tengah upaya hil...,214,166
9,PHE WMO Pasok Kondensat ke Industri Thinner da...,gresik kompas pertamina hulu energi west madur...,gresik pertamina hulu energi west madura offsh...,330,249


In [12]:
# ============================================================
# MENYIMPAN HASIL PREPROCESSING KE GOOGLE DRIVE
# ============================================================
OUTPUT_FILE = "hasil_preprocessing_pertamina.csv"
OUTPUT_PATH = f"/content/drive/MyDrive/ProjectA-PBA/{OUTPUT_FILE}"

df[['Link', 'Judul', 'Isi Berita', 'Status', 'Tag', 'Sentimen', 'Penerbit',
    'final_text_clean', 'final_text_stemmed', 'jumlah_token_before_sw', 'jumlah_token_after_sw']].to_csv(OUTPUT_PATH, index=False)

print(f"Dataset final berhasil disimpan: {OUTPUT_PATH}")

Dataset final berhasil disimpan: /content/drive/MyDrive/ProjectA-PBA/hasil_preprocessing_pertamina.csv
